In [1]:
import pandas as pd
import numpy as np

record=pd.read_csv("Titanic-Dataset.csv")

# quick re-clean
med = record["Age"].median()
record["Age"] = record["Age"].fillna(med)
record = record.drop(columns=["Cabin"])
most_common_port = record["Embarked"].mode()[0]
record["Embarked"] = record["Embarked"].fillna(most_common_port)


In [8]:
# modify an existing column directly
record["Fare"]=record["Fare"].round(2)
record["Sex"]=record["Sex"].replace({"male":"M","female":"F"})
print(record["Sex"].unique())

# update using .loc - change a specific value based on a condition
record.loc[record["Fare"]==0,"Fare"]=record["Fare"].median()
print(record["Fare"].describe())
print(record.loc[record["Fare"]==record["Fare"].median()])

<StringArray>
['M', 'F']
Length: 2, dtype: str
count    891.000000
mean      32.447632
std       49.570245
min        4.010000
25%        7.920000
50%       14.450000
75%       31.000000
max      512.330000
Name: Fare, dtype: float64
     PassengerId  Survived  Pclass                                     Name  \
73            74         0       3              Chronopoulos, Mr. Apostolos   
111          112         0       3                     Zabour, Miss. Hileni   
179          180         0       3                      Leonard, Mr. Lionel   
240          241         0       3                    Zabour, Miss. Thamine   
263          264         0       1                    Harrison, Mr. William   
271          272         1       3             Tornquist, Mr. William Henry   
277          278         0       2              Parkes, Mr. Francis "Frank"   
302          303         0       3          Johnson, Mr. William Cahoone Jr   
362          363         0       3          Barbara, Mr

In [10]:
# create age categories using pd.cut()
record["AgeGroup"]=pd.cut(record["Age"],
                          bins=[0,12,18,35,60,100],
                          labels=["Child","Teen","Adult","MiddleAge","Senior"])
print(record[["Age","AgeGroup"]].head(10))
print(record["AgeGroup"].value_counts())

    Age   AgeGroup
0  22.0      Adult
1  38.0  MiddleAge
2  26.0      Adult
3  35.0      Adult
4  35.0      Adult
5  28.0      Adult
6  54.0  MiddleAge
7   2.0      Child
8  27.0      Adult
9  14.0       Teen
AgeGroup
Adult        535
MiddleAge    195
Teen          70
Child         69
Senior        22
Name: count, dtype: int64


In [ ]:
record["FamilySize"]=record["SibSp"]+record["Parch"] + 1   # +1 to include the passenger themself
record["IsAlone"]=(record["FamilySize"]==1).astype("int")
print(record[["SibSp","Parch","FamilySize","IsAlone"]].head(10))


   SibSp  Parch  FamilySize  IsAlone
0      1      0           2        0
1      1      0           2        0
2      0      0           1        1
3      1      0           2        0
4      0      0           1        1
5      0      0           1        1
6      0      0           1        1
7      3      1           5        0
8      0      2           3        0
9      1      0           2        0


In [25]:
#extract the title (Mr, Mrs, Miss, etc.) from the Name column using string splitting/regex
record["Title"]=record["Name"].str.extract(r",\s*([^\.]*)\.")
print(record["Title"].value_counts())
record["Title"]=record["Title"].replace(["Lady", "Countess", "Capt", "Col", "Don", "Dr", "Major", "Rev", "Sir", "Jonkheer","the Countess"],"Rare")
record["Title"]=record["Title"].replace(["Mlle", "Ms"],"Miss")
record["Title"]=record["Title"].replace("Mme", "Mrs")

print(record["Title"].value_counts())

Title
Mr              517
Miss            182
Mrs             125
Master           40
Dr                7
Rev               6
Major             2
Mlle              2
Col               2
Don               1
Mme               1
Ms                1
Lady              1
Sir               1
Capt              1
the Countess      1
Jonkheer          1
Name: count, dtype: int64
Title
Mr        517
Miss      185
Mrs       126
Master     40
Rare       23
Name: count, dtype: int64


In [21]:
# simulate a "profit" column and salary-style bins
record["Profit"]=record["Fare"]*2-10  # made-up example calc, just for practice
record["FareLevel"]=pd.cut(record["Fare"],
                           bins=[-1,10,30,100],
                           labels=["Low","Mid","High"])

print(record[["Fare","FareLevel"]].head(10))

# conditional column with np.where 
print(record["Fare"].median())
record["HighFare"]=np.where(record["Fare"]>record["Fare"].median(),"Above Median","Below Median")
print(record[["Fare","HighFare"]].head(10))

    Fare FareLevel
0   7.25       Low
1  71.28      High
2   7.92       Low
3  53.10      High
4   8.05       Low
5   8.46       Low
6  51.86      High
7  21.08       Mid
8  11.13       Mid
9  30.07      High
14.45
    Fare      HighFare
0   7.25  Below Median
1  71.28  Above Median
2   7.92  Below Median
3  53.10  Above Median
4   8.05  Below Median
5   8.46  Below Median
6  51.86  Above Median
7  21.08  Above Median
8  11.13  Below Median
9  30.07  Above Median


In [22]:
# once you've extracted useful features, you can drop the raw columns that are less useful now
record_final=record.drop(columns=["Name","Ticket"])
print(record_final.columns)

Index(['PassengerId', 'Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch',
       'Fare', 'Embarked', 'AgeGroup', 'FamilySize', 'IsAlone', 'Title',
       'Profit', 'FareLevel', 'HighFare'],
      dtype='str')


In [26]:
print(record.groupby("Title")["Survived"].mean().sort_values(ascending=False))

Title
Mrs       0.793651
Miss      0.702703
Master    0.575000
Rare      0.347826
Mr        0.156673
Name: Survived, dtype: float64
